In [26]:
import pandas as pd
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

### Load dataset ###

In [27]:
data = pd.read_csv("16k_Movies.csv")

In [28]:
data.shape

(16290, 10)

In [29]:
data.head()

,Unnamed: 0,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres
0,0,Dekalog (1988),"Mar 22, 1996",This masterwork by Krzysztof Kieślowski is one...,7.4,118,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz",9 h 32 m,Drama
1,1,Three Colors: Red,"Nov 23, 1994",Krzysztof Kieslowski closes his Three Colors t...,8.3,241,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz, Ag...",1 h 39 m,"Drama,Mystery,Romance"
2,2,The Conformist,"Oct 22, 1970","Set in Rome in the 1930s, this re-release of B...",7.3,106,Bernardo Bertolucci,"Alberto Moravia, Bernardo Bertolucci",1 h 47 m,Drama
3,3,Tokyo Story,"Mar 13, 1972",Yasujiro Ozu’s Tokyo Story follows an aging co...,8.1,147,Yasujirô Ozu,"Kôgo Noda, Yasujirô Ozu",2 h 16 m,Drama
4,4,The Leopard (re-release),"Aug 13, 2004","Set in Sicily in 1860, Luchino Visconti's spec...",7.8,85,Luchino Visconti,"Giuseppe Tomasi di Lampedusa, Suso Cecchi D'Am...",3 h 7 m,"Drama,History"


### Check missing values ###

In [30]:
data.describe()

,Unnamed: 0,Rating
count,16290.000000,12846.000000
mean,8144.500000,6.617632
std,4702.662278,1.415272
min,0.000000,0.300000
25%,4072.250000,5.800000
50%,8144.500000,6.800000
75%,12216.750000,7.600000
max,16289.000000,10.000000


In [31]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16290 entries, 0 to 16289
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           16290 non-null  int64  
 1   Title                16290 non-null  object 
 2   Release Date         16290 non-null  object 
 3   Description          16290 non-null  object 
 4   Rating               12846 non-null  float64
 5   No of Persons Voted  12829 non-null  object 
 6   Directed by          16283 non-null  object 
 7   Written by           15327 non-null  object 
 8   Duration             16277 non-null  object 
 9   Genres               16285 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 1.2+ MB


In [32]:
print("Columns in your dataset:")
print(data.columns.tolist())
print("\nMissing values per column:")
print(data.isnull().sum())

Columns in your dataset:
['Unnamed: 0', 'Title', 'Release Date', 'Description', 'Rating', 'No of Persons Voted', 'Directed by', 'Written by', 'Duration', 'Genres']

Missing values per column:
Unnamed: 0                0
Title                     0
Release Date              0
Description               0
Rating                 3444
No of Persons Voted    3461
Directed by               7
Written by              963
Duration                 13
Genres                    5
dtype: int64


### Clean dataset ###

keeping the first occurrence

In [33]:
data = data.drop_duplicates(subset=['Title'], keep='first')
print(f"Shape after removing duplicates: {data.shape}")

Shape after removing duplicates: (14693, 10)


Drop rows with missing essential information

In [34]:
data = data.dropna(subset=['Genres'])
print(f"Shape after dropping missing genres: {data.shape}")

Shape after dropping missing genres: (14688, 10)


Fill missing values with empty strings

In [35]:
data['Directed by'] = data['Directed by'].fillna('')
data['Written by'] = data['Written by'].fillna('')
data['Description'] = data['Description'].fillna('')

Clean the Genres column (remove quotes and clean up)

In [36]:
def clean_genres(genres):
    if isinstance(genres, str):
        
        genres = genres.replace('"', '')
        
        genres_list = [g.strip() for g in genres.split(',')]
        return ' '.join(genres_list)
    return ''
data['Genres'] = data['Genres'].apply(clean_genres)

Clean the Duration column (convert to minutes for numerical feature)

In [37]:
def duration_to_minutes(duration):
    if pd.isna(duration):
        return 0
    duration_str = str(duration)
    hours = 0
    minutes = 0
    
    hour_match = re.search(r'(\d+)\s*h', duration_str)
    if hour_match:
        hours = int(hour_match.group(1))
    
    min_match = re.search(r'(\d+)\s*m', duration_str)
    if min_match:
        minutes = int(min_match.group(1))
    
    return hours * 60 + minutes

data['Duration_minutes'] = data['Duration'].apply(duration_to_minutes)

data = data.dropna(subset=['Duration_minutes'])

Clean Rating (convert to float)

In [38]:
data['Rating_clean'] = pd.to_numeric(data['Rating'], errors='coerce').fillna('')

Clean number of votes

In [39]:
data['Votes_clean'] = pd.to_numeric(data['No of Persons Voted'], errors='coerce').fillna(0)

In [40]:
data.isnull().sum()

Unnamed: 0                0
Title                     0
Release Date              0
Description               0
Rating                 3168
No of Persons Voted    3182
Directed by               0
Written by                0
Duration                 12
Genres                    0
Duration_minutes          0
Rating_clean              0
Votes_clean               0
dtype: int64

In [41]:
data.head()

,Unnamed: 0,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres,Duration_minutes,Rating_clean,Votes_clean
0,0,Dekalog (1988),"Mar 22, 1996",This masterwork by Krzysztof Kieślowski is one...,7.4,118,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz",9 h 32 m,Drama,572,7.4,118.0
1,1,Three Colors: Red,"Nov 23, 1994",Krzysztof Kieslowski closes his Three Colors t...,8.3,241,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz, Ag...",1 h 39 m,Drama Mystery Romance,99,8.3,241.0
2,2,The Conformist,"Oct 22, 1970","Set in Rome in the 1930s, this re-release of B...",7.3,106,Bernardo Bertolucci,"Alberto Moravia, Bernardo Bertolucci",1 h 47 m,Drama,107,7.3,106.0
3,3,Tokyo Story,"Mar 13, 1972",Yasujiro Ozu’s Tokyo Story follows an aging co...,8.1,147,Yasujirô Ozu,"Kôgo Noda, Yasujirô Ozu",2 h 16 m,Drama,136,8.1,147.0
4,4,The Leopard (re-release),"Aug 13, 2004","Set in Sicily in 1860, Luchino Visconti's spec...",7.8,85,Luchino Visconti,"Giuseppe Tomasi di Lampedusa, Suso Cecchi D'Am...",3 h 7 m,Drama History,187,7.8,85.0


In [42]:
data.shape

(14688, 13)

### Feature Engineering ###

Combine features for TF-IDF

In [43]:
data['combined_features'] = (data['Genres'] + ' ' + data['Directed by'] + ' ' + data['Written by'])

In [44]:
data.head()

,Unnamed: 0,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres,Duration_minutes,Rating_clean,Votes_clean,combined_features
0,0,Dekalog (1988),"Mar 22, 1996",This masterwork by Krzysztof Kieślowski is one...,7.4,118,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz",9 h 32 m,Drama,572,7.4,118.0,Drama Krzysztof Kieslowski Krzysztof Kieslowsk...
1,1,Three Colors: Red,"Nov 23, 1994",Krzysztof Kieslowski closes his Three Colors t...,8.3,241,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz, Ag...",1 h 39 m,Drama Mystery Romance,99,8.3,241.0,Drama Mystery Romance Krzysztof Kieslowski Krz...
2,2,The Conformist,"Oct 22, 1970","Set in Rome in the 1930s, this re-release of B...",7.3,106,Bernardo Bertolucci,"Alberto Moravia, Bernardo Bertolucci",1 h 47 m,Drama,107,7.3,106.0,"Drama Bernardo Bertolucci Alberto Moravia, Ber..."
3,3,Tokyo Story,"Mar 13, 1972",Yasujiro Ozu’s Tokyo Story follows an aging co...,8.1,147,Yasujirô Ozu,"Kôgo Noda, Yasujirô Ozu",2 h 16 m,Drama,136,8.1,147.0,"Drama Yasujirô Ozu Kôgo Noda, Yasujirô Ozu"
4,4,The Leopard (re-release),"Aug 13, 2004","Set in Sicily in 1860, Luchino Visconti's spec...",7.8,85,Luchino Visconti,"Giuseppe Tomasi di Lampedusa, Suso Cecchi D'Am...",3 h 7 m,Drama History,187,7.8,85.0,Drama History Luchino Visconti Giuseppe Tomasi...


Initialize TF-IDF Vectorizer

In [45]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

In [46]:
tfidf_matrix = tfidf_vectorizer.fit_transform(data['combined_features'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (14688, 5000)


Compute cosine similarity

In [47]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

Cosine similarity matrix shape: (14688, 14688)


In [48]:
scaler = MinMaxScaler()
data['popularity_score'] = scaler.fit_transform(data[['Votes_clean']])

In [49]:
model_artifacts = {
    'tfidf_vectorizer': tfidf_vectorizer,
    'cosine_sim': cosine_sim,
    'data': data,
    'scaler': scaler
}

with open('movie_recommender.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("Model saved successfully!")

Model saved successfully!


In [50]:
data.head()

,Unnamed: 0,Title,Release Date,Description,Rating,No of Persons Voted,Directed by,Written by,Duration,Genres,Duration_minutes,Rating_clean,Votes_clean,combined_features,popularity_score
0,0,Dekalog (1988),"Mar 22, 1996",This masterwork by Krzysztof Kieślowski is one...,7.4,118,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz",9 h 32 m,Drama,572,7.4,118.0,Drama Krzysztof Kieslowski Krzysztof Kieslowsk...,0.118474
1,1,Three Colors: Red,"Nov 23, 1994",Krzysztof Kieslowski closes his Three Colors t...,8.3,241,Krzysztof Kieslowski,"Krzysztof Kieslowski, Krzysztof Piesiewicz, Ag...",1 h 39 m,Drama Mystery Romance,99,8.3,241.0,Drama Mystery Romance Krzysztof Kieslowski Krz...,0.241968
2,2,The Conformist,"Oct 22, 1970","Set in Rome in the 1930s, this re-release of B...",7.3,106,Bernardo Bertolucci,"Alberto Moravia, Bernardo Bertolucci",1 h 47 m,Drama,107,7.3,106.0,"Drama Bernardo Bertolucci Alberto Moravia, Ber...",0.106426
3,3,Tokyo Story,"Mar 13, 1972",Yasujiro Ozu’s Tokyo Story follows an aging co...,8.1,147,Yasujirô Ozu,"Kôgo Noda, Yasujirô Ozu",2 h 16 m,Drama,136,8.1,147.0,"Drama Yasujirô Ozu Kôgo Noda, Yasujirô Ozu",0.147590
4,4,The Leopard (re-release),"Aug 13, 2004","Set in Sicily in 1860, Luchino Visconti's spec...",7.8,85,Luchino Visconti,"Giuseppe Tomasi di Lampedusa, Suso Cecchi D'Am...",3 h 7 m,Drama History,187,7.8,85.0,Drama History Luchino Visconti Giuseppe Tomasi...,0.085341
